# 🔍 Semantic Search with FAISS

**Source:** [HuggingFace LLM Course – Chapter 5, Section 6](https://huggingface.co/learn/llm-course/chapter5/6)

In this notebook we build a **semantic search engine** over GitHub issues from the 🤗 Datasets repository. Instead of matching keywords, we use **dense embeddings** from a Transformer model to find issues that are semantically similar to a user's question.

## 🗺️ Workflow Overview
1. Load and filter the GitHub issues dataset
2. Clean and reshape the data (explode comments, remove noise)
3. Concatenate issue fields into a single searchable `text` field
4. Generate 768-dimensional embeddings using `multi-qa-mpnet-base-dot-v1`
5. Build a FAISS index for fast nearest-neighbor search
6. Query the index with a natural language question and retrieve the top results

---

## 🛠️ Step 1: Install Required Libraries

Before we start, we need to install the libraries used in this notebook:

- **`datasets`** – The Hugging Face Datasets library for loading and processing datasets.
- **`evaluate`** – For evaluation metrics.
- **`transformers[sentencepiece]`** – Hugging Face Transformers with SentencePiece support for tokenization.
- **`faiss-gpu`** – Facebook AI Similarity Search (FAISS) for fast vector similarity search (GPU version for speed).

> 💡 Uncomment the lines above if running this notebook for the first time.

In [2]:
#!pip install datasets evaluate transformers[sentencepiece]
#!pip install faiss-gpu

## 📥 Step 2: Load the GitHub Issues Dataset

We load the GitHub issues dataset (collected in the previous section of the course) directly from the Hugging Face Hub using `load_dataset()`.

- **`split="train"`** – Loads only the training split, which gives us a `Dataset` object instead of a `DatasetDict`.
- The dataset contains 5,000 GitHub issues and pull requests from the 🤗 Datasets repository, each with fields like `title`, `body`, `comments`, `html_url`, and `is_pull_request`.

Our goal is to build a **semantic search engine** over these issues so we can find relevant answers to user questions.

In [3]:
from datasets import load_dataset

issues_dataset = load_dataset("SAK-07/github-issues", split="train")
issues_dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 5000
})

## 🔍 Step 3: Filter Issues (Remove Pull Requests & Empty Comments)

We filter the raw dataset to keep only items that are relevant for our search engine:

- **`is_pull_request == False`** – We exclude pull requests because they typically don't contain user question/answer pairs.
- **`len(x["comments"]) > 0`** – We exclude issues with no comments since they offer no answers to search over.

After filtering, the dataset shrinks from **5,000** rows to **~1,893** rows — these are genuine issues with at least one comment.

In [4]:
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 1893
})

## ✂️ Step 4: Keep Only Relevant Columns

Most of the columns in the raw dataset are not useful for our semantic search engine. We use `symmetric_difference()` to compute the set of columns to drop, then call `remove_columns()` to clean up the dataset.

We keep only **4 columns**:
| Column | Purpose |
|--------|---------|
| `title` | Issue title — gives context about the problem |
| `body` | Issue description — contains the full question |
| `comments` | List of comments — these are our potential answers |
| `html_url` | GitHub URL — lets us link back to the original issue |

In [5]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 1893
})

## 🐼 Step 5: Convert to Pandas DataFrame

We switch the dataset to **Pandas format** using `set_format("pandas")`, then load all rows into a DataFrame with `df = issues_dataset[:]`.

This is necessary because we need to use the Pandas `.explode()` method in the next step to "flatten" the `comments` column — each issue currently has a **list** of comments, and we want **one comment per row**.

In [6]:
issues_dataset.set_format("pandas")
df = issues_dataset[:]

## 👀 Step 6: Inspect Comments for the First Issue

Let's peek at what the `comments` column actually looks like. For the first issue, `.tolist()` reveals that it stores a **list of comment strings**.

Each entry in this list is a full comment text written by a GitHub user. In the next step, we'll "explode" these lists so every comment gets its own row in the dataset.

In [7]:
df["comments"][0].tolist()

["Hey, ran into something similar a while back when we were packaging a custom dataset loader for a client at v4.7.x. Turns out, PyPI distributions sometimes skip test files and fixtures like `conftest.py` since they're not critical for runtime—honestly, it's a packaging config thing in `setup.py` or `pyproject.toml`. We ended up manually pulling the source from GitHub for testing purposes when we needed those files. Stick to the repo if you’re debugging or running tests locally."]

## 💥 Step 7: Explode the Comments Column

We use **`DataFrame.explode()`** to transform the dataset so that each row contains exactly **one comment**.

- Before: each row = one GitHub issue with a *list* of comments.
- After: each row = one (issue, single comment) pair — all other columns (`title`, `body`, `html_url`) are replicated for each comment.

This is essential so we can later embed and search over **individual comments** rather than lists.

In [8]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)


,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,"tests/conftest.py, tests/_test_patching.py, te...","Hey, ran into something similar a while back w...",### Describe the bug\n\nVersion: 4.8.5\n\n\n##...
1,https://github.com/huggingface/datasets/issues...,[Optimization] Prevent per-thread instantiatio...,Is it open for contribution?,### Feature request\n\nModify the dataset load...
2,https://github.com/huggingface/datasets/issues...,[Optimization] Prevent per-thread instantiatio...,I think arbitrary filesystems would be recreat...,### Feature request\n\nModify the dataset load...
3,https://github.com/huggingface/datasets/issues...,`.map()` on a streaming IterableDataset silent...,I think this line here is causing issues: \nht...,### Describe the bug\n\nAfter `ds.map(fn)` on ...


## 🔄 Step 8: Convert Back to Hugging Face Dataset

Now that we've exploded the comments into individual rows using Pandas, we convert the DataFrame back to a Hugging Face `Dataset` using `Dataset.from_pandas()`.

The result is a dataset with **~7,175 rows** — one row per (issue, comment) pair. We're back to the efficient Hugging Face `Dataset` format, which supports fast `.map()`, `.filter()`, and FAISS indexing.

In [9]:
from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 7175
})

## 📏 Step 9: Compute Comment Length

We use `Dataset.map()` to add a new column called **`comment_length`**, which stores the **word count** of each comment.

This is computed by splitting the comment string on whitespace with `.split()` and measuring the length of the resulting list. We'll use this column in the next step to filter out very short comments that are unlikely to be useful answers.

In [10]:
comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)

Map:   0%|          | 0/7175 [00:00<?, ? examples/s]

## 🧹 Step 10: Filter Out Short Comments

Short comments like *"Thanks!"* or *"cc @someone"* don't add useful information for a search engine. We filter to keep only comments with **more than 15 words**.

After filtering:
- **Before:** ~7,175 rows
- **After:** ~5,229 rows

> The threshold of 15 words is a reasonable heuristic — feel free to experiment with different values!

In [11]:
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

Filter:   0%|          | 0/7175 [00:00<?, ? examples/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 5229
})

## 🔗 Step 11: Concatenate Title, Body, and Comment into One Text Field

For embedding, we want each row to be represented as a **single rich text string** that captures the full context of the issue. We define a function `concatenate_text()` that joins the `title`, `body`, and `comments` fields together with newline separators.

```
text = title + " \n " + body + " \n " + comment
```

This combined `text` field is what we'll feed to our embedding model. By including the title and body alongside each comment, the model has full context about what the issue is about — not just the comment in isolation.

We also use `or ""` guards to safely handle any `None` values in the dataset.

In [12]:
def concatenate_text(examples):
    return {
        "text": (examples["title"] or "")
        + " \n "
        + (examples["body"] or "")
        + " \n "
        + (examples["comments"] or "")
    }

comments_dataset = comments_dataset.map(concatenate_text)

Map:   0%|          | 0/5229 [00:00<?, ? examples/s]

## 🤖 Step 12: Load the Embedding Model & Tokenizer

We use the **`sentence-transformers/multi-qa-mpnet-base-dot-v1`** checkpoint — a model specifically trained for **asymmetric semantic search**, which is ideal for our use case:

- **Asymmetric search** = a *short query* (e.g. a question) is matched against *longer documents* (e.g. issue comments).

We load both the **tokenizer** and the **model** using Hugging Face's `AutoTokenizer` and `AutoModel` classes.

| Component | Purpose |
|-----------|--------|
| `AutoTokenizer` | Converts text strings into token IDs the model understands |
| `AutoModel` | The MPNet transformer that produces contextual token embeddings |

> This model produces **768-dimensional** embedding vectors for each input text.

In [13]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## ⚡ Step 13: Move Model to GPU

We move the model to the **GPU** using `model.to(device)` to significantly speed up the embedding process.

- PyTorch's `torch.device("cuda")` selects the GPU if available.
- All input tensors also need to be moved to the same device before being passed to the model.

> If you're running this on a CPU-only machine, change `"cuda"` to `"cpu"` — it will just be slower.

In [14]:
import torch

device = torch.device("cuda")
model.to(device)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

## 🏊 Step 14: Define CLS Pooling

Transformer models output a hidden state for every token in the input. To get a **single vector** representing the entire text, we need a pooling strategy.

We use **CLS pooling**: extract the hidden state corresponding to the special `[CLS]` token (always the first token at index `0`). This token is specifically trained to capture the overall meaning of the input sequence.

```python
# model_output.last_hidden_state has shape: [batch_size, seq_len, hidden_dim]
# [:, 0] selects the [CLS] token for all items in the batch
return model_output.last_hidden_state[:, 0]
```

In [15]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

## 🔢 Step 15: Define the Embedding Function

We define `get_embeddings()`, a helper function that takes a list of text strings and returns their **embedding vectors**. Here's what it does step-by-step:

1. **Tokenize** — Convert text strings to token IDs, with padding and truncation to handle variable lengths.
2. **Move to GPU** — Transfer all input tensors to the GPU device.
3. **Run the model** — Forward pass through the MPNet model to get hidden states.
4. **Pool** — Apply CLS pooling to get one vector per input text.

The output shape is `[batch_size, 768]` — one 768-dimensional vector per text.

In [16]:
def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

## ✅ Step 16: Verify Embedding Shape

Let's test our `get_embeddings()` function on the first text entry in our dataset to verify it works correctly.

- **Input:** A single text string (title + body + comment)
- **Output shape:** `torch.Size([1, 768])` — one vector of 768 dimensions

This confirms the pipeline works correctly before we run it over the entire dataset.

In [17]:
embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

torch.Size([1, 768])

## 🗺️ Step 17: Generate Embeddings for All Comments

Now we apply `get_embeddings()` to every row in the dataset using `Dataset.map()` to create a new **`embeddings`** column.

Key details:
- `.detach()` — Detaches the tensor from the computation graph (no gradient tracking needed at inference time).
- `.cpu().numpy()` — Moves the tensor back to CPU and converts to a NumPy array, which is required by the FAISS indexer.
- `[0]` — Selects the single vector from the batch dimension since we process one row at a time.

> ⏳ This step can take several minutes since it embeds all 5,229 comments through a neural network!

In [18]:
embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Map:   0%|          | 0/5229 [00:00<?, ? examples/s]

## 📦 Step 18: Install FAISS (CPU Version)

If you're running this on a CPU-only machine (no GPU), you can install the **CPU version of FAISS**.

- **`faiss-gpu`** — Faster, for machines with a CUDA-compatible GPU (installed at the top).
- **`faiss-cpu`** — Slower but works anywhere. Uncomment this line if needed.

FAISS (Facebook AI Similarity Search) is a highly optimized library for efficiently searching through large collections of embedding vectors.

In [19]:
#!pip install faiss-cpu

## 🗂️ Step 19: Build the FAISS Index

We use `Dataset.add_faiss_index()` to build a **FAISS index** over our `embeddings` column. This creates a special data structure that enables **blazing-fast nearest-neighbor search** over our 5,229 comment vectors.

**How FAISS works:**
- It organizes the embedding vectors in a way that avoids comparing a query against *every single* vector.
- Instead, it uses smart indexing (like clustering and quantization) to find the most similar vectors in milliseconds.

After this step, `embeddings_dataset` is augmented with an in-memory FAISS index — ready for semantic querying!

In [20]:
embeddings_dataset.add_faiss_index(column="embeddings")

  0%|          | 0/6 [00:00<?, ?it/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length', 'text', 'embeddings'],
    num_rows: 5229
})

## 🔢 Step 20: Verify FAISS Installation

We import `faiss` and print its version number to confirm the library is correctly installed and accessible.

This is a quick sanity check before we run queries against the index.

In [21]:
import faiss
print(faiss.__version__)

1.13.2


## ❓ Step 21: Embed a Search Query

Now for the exciting part — semantic search! We define a natural language **question** and embed it using the same `get_embeddings()` function we used for the corpus.

```
question = "How can I load a dataset offline?"
```

The result is a **768-dimensional query vector** — in the exact same embedding space as our corpus vectors. This is crucial: because both the question and the documents were embedded with the same model, their vectors are directly comparable using dot-product similarity.

In [23]:
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

(1, 768)

## 🔎 Step 22: Retrieve the Top-K Most Similar Documents

We use `Dataset.get_nearest_examples()` to search the FAISS index for the **5 most semantically similar** comments to our query.

- **`"embeddings"`** — The name of the indexed column.
- **`question_embedding`** — Our query vector.
- **`k=5`** — Return the top 5 nearest neighbors.

The function returns:
- **`scores`** — Similarity scores (higher = more similar). These are dot-product scores.
- **`samples`** — A dict containing all columns (`title`, `body`, `comments`, `html_url`) for the 5 best matches.

In [24]:
scores, samples = embeddings_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

## 📊 Step 23: Organize Results in a DataFrame

We collect the retrieved samples into a **Pandas DataFrame** and attach the similarity `scores` as a new column.

Then we sort by `scores` in **descending order** (`ascending=False`) so the most relevant result appears first.

This makes it easy to iterate through the results and inspect the best matches.

In [25]:
import pandas as pd

samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

## 🎯 Step 24: Display the Search Results

Finally, we iterate over the sorted results and print the key information for each match:

| Field | Description |
|-------|-------------|
| `COMMENT` | The actual comment text that matched our query |
| `SCORE` | Dot-product similarity score (higher = better match) |
| `TITLE` | The GitHub issue title for context |
| `URL` | Link to the original GitHub issue |

The results demonstrate the power of **semantic search** — the engine understands the *meaning* of the question ("load a dataset offline") and finds relevant answers even if they use different words than the query.

> 💡 **Try it out!** Change the `question` in the previous cell to something else and re-run. You can also increase `k=5` to retrieve more results.

In [26]:
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

COMMENT: Hi @ssuwelack I tried loading a HF dataset with your viewer but got this error https://github.com/Renumics/spotlight/issues/461 hope the team can help me on this. Thanks!
SCORE: 34.73320770263672
TITLE: Offline dataset viewer
URL: https://github.com/huggingface/datasets/issues/6139

COMMENT: Feel free to use `dataset.save_to_disk(...)`, then scp the directory containing the saved dataset and reload it on your other machine using `dataset = load_from_disk(...)`
SCORE: 34.543357849121094
TITLE: How to load existing downloaded dataset ?
URL: https://github.com/huggingface/datasets/issues/6387

COMMENT: Hi, we are building an offline dataset viewer: https://github.com/Renumics/spotlight
It supports many HF datasets, but currently you have to use it via Pandas:
df=ds.to_pandas()
spotlight.show(df)

Would love to hear from you if that works for your use case. If not, feel free to open an issue on the repo: https://github.com/Renumics/spotlight/issues
SCORE: 33.191856384277344
TITLE: